In [1]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

without understanding tokens and embeddings we cannot understand how LLMs works see fig 2-1

LLM TOKENIZATION

the model generates its response in one token at a time, but tokens arent only the outputs but also the input which the model sees, a text form is broken into tokens before sending it to the model

HOW tokenizers prepare the inputs for LLMs

if we see  a high level view LLMS take an input prompt and then generate a response
but thats not what happens inside, the input prompt has to go through a tokenizer which breaks the input into pieces , see fig 2-3

to further understand this lets download an LLM and see how can we tokenize the input prompt



In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

we will first write our prompt then tokenize it and give the tokens to the model, now we will tell the model to generate only 20 new tokens

In [3]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=20

)
# Print the output
print(tokenizer.decode(generation_output[0]))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


everything after subject: is the modles 20 tokens output, the model didnt actually got the raw prompt but the tokenizers took the raw prompt and returned the information needed by the model in the form of input_ids variable, which the model used as its input

In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


this shows the LLMs respond to a series of numbers as shown in fig 2-4, each one is unique id of the specific token, the token can be a word, a char, or part of the word, these ids reference a table inside the tokenizer which contains all the tokens it knows

if we want to see the actual words which these ids hold we can use the decode method by tokenizer

In [5]:
for id in input_ids[0]:
  print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


some tokens are complete words while some are part of words, punctuation char are their own tokens

there are no tokens for spaces and partial tokens like "izing" and "ic" have a special hidden char at their beginning which says they are connected with token that precedes them in text, tokens without that special char are assumed to have a space before them

if we move towards the output side we can also see the tokens generated by model for outputs using generation_output variable, it will show us the input tokens as well as the output tokens

In [6]:
print(generation_output)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')


after 32001 all the tokens are of output, we can see the actual text for output side too using tokenizers decode method

In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:
